In [1]:
import sqlite3
import pandas as pd

In [2]:
db_file = "sqlite_db_pythonsqlite.db"

In [3]:
def run_sql(sql_code):
    conn = sqlite3.connect(db_file)
    # 使用 pandas 让结果显示为整齐的表格
    df = pd.read_sql_query(sql_code, conn)
    conn.close()
    return df

In [4]:
run_sql("SELECT * FROM Facilities LIMIT 5")

,facid,name,membercost,guestcost,initialoutlay,monthlymaintenance
0,0,Tennis Court 1,5.0,25.0,10000,200
1,1,Tennis Court 2,5.0,25.0,8000,200
2,2,Badminton Court,0.0,15.5,4000,50
3,3,Table Tennis,0.0,5.0,320,10
4,4,Massage Room 1,9.9,80.0,4000,3000


In [5]:
# Q1: Some of the facilities charge a fee to members, but some do not.
run_sql("SELECT name FROM Facilities WHERE membercost > 0")

,name
0,Tennis Court 1
1,Tennis Court 2
2,Massage Room 1
3,Massage Room 2
4,Squash Court


In [6]:
# Q2: How many facilities do not charge a fee to members?
run_sql("SELECT COUNT(*) FROM Facilities WHERE membercost = 0")

,COUNT(*)
0,4


In [7]:
# Q3: Write an SQL query to show a list of facilities that charge a fee to members, where the fee is less than 20% of the facility's monthly maintenance cost. Return the facid, facility name, member cost, and monthly maintenance of the facilities in question.
q3_sql = """
SELECT facid, name, membercost, monthlymaintenance
FROM Facilities
WHERE membercost > 0 
  AND membercost < (monthlymaintenance * 0.2);
"""
run_sql(q3_sql)

,facid,name,membercost,monthlymaintenance
0,0,Tennis Court 1,5.0,200
1,1,Tennis Court 2,5.0,200
2,4,Massage Room 1,9.9,3000
3,5,Massage Room 2,9.9,3000
4,6,Squash Court,3.5,80


In [8]:
# Q4: Write an SQL query to retrieve the details of facilities with ID 1 and 5. Try writing the query without using the OR operator.
q4_sql = """
SELECT * FROM Facilities 
WHERE facid IN (1, 5);
"""
run_sql(q4_sql)

,facid,name,membercost,guestcost,initialoutlay,monthlymaintenance
0,1,Tennis Court 2,5.0,25,8000,200
1,5,Massage Room 2,9.9,80,4000,3000


In [9]:
# Q5: Produce a list of facilities, with each labelled as 'cheap' or 'expensive', depending on if their monthly maintenance cost is more than $100. Return the name and monthly maintenance of the facilities in question.
q5_sql = """
SELECT name, monthlymaintenance,
    CASE WHEN monthlymaintenance > 100 THEN 'expensive'
         ELSE 'cheap' END AS label
FROM Facilities;
"""
run_sql(q5_sql)

,name,monthlymaintenance,label
0,Tennis Court 1,200,expensive
1,Tennis Court 2,200,expensive
2,Badminton Court,50,cheap
3,Table Tennis,10,cheap
4,Massage Room 1,3000,expensive
5,Massage Room 2,3000,expensive
6,Squash Court,80,cheap
7,Snooker Table,15,cheap
8,Pool Table,15,cheap


In [10]:
# Q6: You'd like to get the first and last name of the last member(s) who signed up. Try not to use the LIMIT clause for your solution.
q6_sql = """
SELECT firstname, surname
FROM Members
WHERE joindate = (SELECT MAX(joindate) FROM Members);
"""
run_sql(q6_sql)

,firstname,surname
0,Darren,Smith


In [11]:
# Q7: Produce a list of all members who have used a tennis court. Include in your output the name of the court, and the name of the member formatted as a single column. Ensure no duplicate data, and order by the member name.
q7_sql = """
SELECT DISTINCT f.name AS facility, 
                m.firstname || ' ' || m.surname AS member_name
FROM Bookings AS b
INNER JOIN Facilities AS f ON b.facid = f.facid
INNER JOIN Members AS m ON b.memid = m.memid
WHERE f.name LIKE 'Tennis Court%'
ORDER BY member_name;
"""
run_sql(q7_sql)

,facility,member_name
0,Tennis Court 1,Anne Baker
1,Tennis Court 2,Anne Baker
2,Tennis Court 2,Burton Tracy
3,Tennis Court 1,Burton Tracy
4,Tennis Court 1,Charles Owen
5,Tennis Court 2,Charles Owen
6,Tennis Court 2,Darren Smith
7,Tennis Court 1,David Farrell
8,Tennis Court 2,David Farrell
9,Tennis Court 2,David Jones


In [12]:
# Q8: Produce a list of bookings on the day of 2012-09-14 which will cost the member (or guest) more than $30. Remember that guests have different costs to members (the listed costs are per half-hour 'slot'), and the guest user's ID is always 0. Include in your output the name of the facility, the name of the member formatted as a single column, and the cost. Order by descending cost, and do not use any subqueries.
q8_sql = """
SELECT f.name AS facility, 
       m.firstname || ' ' || m.surname AS member_name,
       CASE WHEN b.memid = 0 THEN b.slots * f.guestcost
            ELSE b.slots * f.membercost END AS cost
FROM Bookings AS b
INNER JOIN Facilities AS f ON b.facid = f.facid
INNER JOIN Members AS m ON b.memid = m.memid
WHERE b.starttime LIKE '2012-09-14%'
  AND ((b.memid = 0 AND b.slots * f.guestcost > 30) OR
       (b.memid != 0 AND b.slots * f.membercost > 30))
ORDER BY cost DESC;
"""
run_sql(q8_sql)




,facility,member_name,cost
0,Massage Room 2,GUEST GUEST,320.0
1,Massage Room 1,GUEST GUEST,160.0
2,Massage Room 1,GUEST GUEST,160.0
3,Massage Room 1,GUEST GUEST,160.0
4,Tennis Court 2,GUEST GUEST,150.0
5,Tennis Court 1,GUEST GUEST,75.0
6,Tennis Court 1,GUEST GUEST,75.0
7,Tennis Court 2,GUEST GUEST,75.0
8,Squash Court,GUEST GUEST,70.0
9,Massage Room 1,Jemima Farrell,39.6


In [13]:
# Q9: This time, produce the same result as in Q8, but using a subquery.
q9_sql = """
SELECT sub.facility, sub.member_name, sub.cost
FROM (
    SELECT f.name AS facility, 
           m.firstname || ' ' || m.surname AS member_name,
           CASE WHEN b.memid = 0 THEN b.slots * f.guestcost
                ELSE b.slots * f.membercost END AS cost
    FROM Bookings AS b
    INNER JOIN Facilities AS f ON b.facid = f.facid
    INNER JOIN Members AS m ON b.memid = m.memid
    WHERE b.starttime LIKE '2012-09-14%'
) AS sub
WHERE sub.cost > 30
ORDER BY sub.cost DESC;
"""
run_sql(q9_sql)




,facility,member_name,cost
0,Massage Room 2,GUEST GUEST,320.0
1,Massage Room 1,GUEST GUEST,160.0
2,Massage Room 1,GUEST GUEST,160.0
3,Massage Room 1,GUEST GUEST,160.0
4,Tennis Court 2,GUEST GUEST,150.0
5,Tennis Court 1,GUEST GUEST,75.0
6,Tennis Court 1,GUEST GUEST,75.0
7,Tennis Court 2,GUEST GUEST,75.0
8,Squash Court,GUEST GUEST,70.0
9,Massage Room 1,Jemima Farrell,39.6


In [14]:
#Q10: Produce a list of facilities with a total revenue less than 1000. The output of facility name and total revenue, sorted by revenue. Remember that there's a different cost for guests and members!
q10_sql = """
SELECT f.name, 
       SUM(CASE WHEN b.memid = 0 THEN b.slots * f.guestcost
                ELSE b.slots * f.membercost END) AS revenue
FROM Bookings AS b
INNER JOIN Facilities AS f ON b.facid = f.facid
GROUP BY f.name
HAVING revenue < 1000
ORDER BY revenue;
"""
run_sql(q10_sql)



,name,revenue
0,Table Tennis,180
1,Snooker Table,240
2,Pool Table,270


In [15]:
# Q11: Produce a report of members and who recommended them in alphabetic surname,firstname order
q11_sql = """
SELECT m1.surname AS mem_surname, 
       m1.firstname AS mem_firstname, 
       m2.surname AS rec_surname, 
       m2.firstname AS rec_firstname
FROM Members AS m1
LEFT JOIN Members AS m2 ON m1.recommendedby = m2.memid
WHERE m1.recommendedby IS NOT NULL AND m1.recommendedby != ''
ORDER BY m2.surname, m2.firstname;
"""
run_sql(q11_sql)


,mem_surname,mem_firstname,rec_surname,rec_firstname
0,Sarwin,Ramnaresh,Bader,Florence
1,Coplin,Joan,Baker,Timothy
2,Genting,Matthew,Butters,Gerald
3,Baker,Timothy,Farrell,Jemima
4,Pinker,David,Farrell,Jemima
5,Rumney,Henrietta,Genting,Matthew
6,Jones,Douglas,Jones,David
7,Dare,Nancy,Joplette,Janice
8,Jones,David,Joplette,Janice
9,Hunt,John,Purview,Millicent


In [16]:
# Q12: Find the facilities with their usage by member, but not guests
q12_sql = """
SELECT f.name, SUM(b.slots) AS member_usage
FROM Bookings AS b
INNER JOIN Facilities AS f ON b.facid = f.facid
WHERE b.memid != 0
GROUP BY f.name
ORDER BY member_usage DESC;
"""
run_sql(q12_sql)

,name,member_usage
0,Badminton Court,1086
1,Tennis Court 1,957
2,Massage Room 1,884
3,Tennis Court 2,882
4,Snooker Table,860
5,Pool Table,856
6,Table Tennis,794
7,Squash Court,418
8,Massage Room 2,54


In [17]:
# Q13: Find the facilities usage by month, but not guests
q13_sql = """
SELECT f.name, 
       strftime('%m', b.starttime) AS month, 
       SUM(b.slots) AS usage
FROM Bookings AS b
INNER JOIN Facilities AS f ON b.facid = f.facid
WHERE b.memid != 0
GROUP BY f.name, month
ORDER BY month, usage DESC;
"""
run_sql(q13_sql)

,name,month,usage
0,Tennis Court 1,07,201
1,Massage Room 1,07,166
2,Badminton Court,07,165
3,Snooker Table,07,140
4,Tennis Court 2,07,123
5,Pool Table,07,110
6,Table Tennis,07,98
7,Squash Court,07,50
8,Massage Room 2,07,8
9,Badminton Court,08,414
